In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Nitin Phanse\AppData\Local\Temp\ipykernel_6540\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory): # takes the raw text from your PDFs and converts it into a list of Langchain Document objects.

    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf")) # this is a regular expression to get all the pdf files

    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f" X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("data/pdf")

Found 7 PDF files to process

Processing: anthropic- constitutional ai.pdf
 ✓ Loaded 34 pages

Processing: attention is all you need.pdf
 ✓ Loaded 15 pages

Processing: GPT-2.pdf
 ✓ Loaded 24 pages

Processing: GPT-3.pdf
 ✓ Loaded 75 pages

Processing: GPT.pdf
 ✓ Loaded 12 pages

Processing: InstructGPT.pdf


Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 95 0 (offset 0)


 ✓ Loaded 68 pages

Processing: NeurIPS-2023-direct-preference-optimization-your-language-model-is-secretly-a-reward-model-Paper-Conference.pdf
 ✓ Loaded 14 pages

Total documents loaded: 242


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-12-19T01:38:58+00:00', 'author': '', 'keywords': '', 'moddate': '2022-12-19T01:38:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\anthropic- constitutional ai.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'anthropic- constitutional ai.pdf', 'file_type': 'pdf'}, page_content='Constitutional AI: Harmlessness from AI Feedback\nYuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion,\nAndy Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon,\nCarol Chen, Catherine Olsson, Christopher Olah, Danny Hernandez, Dawn Drain,\nDeep Ganguli, Dustin Li, Eli Tran-Johnson, Ethan Perez, Jamie Kerr, Jared Mueller,\nJeffrey Ladish, Joshua Landau, Kamal Ndousse, Kamile Lukosuite, Liane Lovitt,\nMichael 

### CHUNKING

In [4]:
### Text splitting get into chunks
### We are using: "Recursive Character Text Splitter" here

# Define a function to break large documents into smaller, overlapping chunks for a Vector DB/RAG pipeline
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    
    # Initialize the text splitter with specific rules on how to break the text
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,       # Maximum number of characters in a single chunk
        chunk_overlap=chunk_overlap, # Amount of characters to overlap between consecutive chunks to maintain context
        length_function=len,         # Function used to calculate the length of the text
        separators=["\n\n", "\n", " ", ""] # Hierarchy of separators to split by (paragraphs first, then newlines, spaces, characters)
    )
    
    # Execute the splitting process on the provided list of Document objects
    split_docs = text_splitter.split_documents(documents)
    
    # Output the results showing how many new chunks were generated from the original pages
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    # If chunks were successfully created, print a preview of the first one to verify it worked
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)

Split 242 documents into 1020 chunks

Example chunk:
Content: Constitutional AI: Harmlessness from AI Feedback
Yuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion,
Andy Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon,
...
Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-12-19T01:38:58+00:00', 'author': '', 'keywords': '', 'moddate': '2022-12-19T01:38:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\anthropic- constitutional ai.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'anthropic- constitutional ai.pdf', 'file_type': 'pdf'}


In [6]:
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-12-19T01:38:58+00:00', 'author': '', 'keywords': '', 'moddate': '2022-12-19T01:38:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\anthropic- constitutional ai.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'anthropic- constitutional ai.pdf', 'file_type': 'pdf'}, page_content='Constitutional AI: Harmlessness from AI Feedback\nYuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion,\nAndy Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon,\nCarol Chen, Catherine Olsson, Christopher Olah, Danny Hernandez, Dawn Drain,\nDeep Ganguli, Dustin Li, Eli Tran-Johnson, Ethan Perez, Jamie Kerr, Jared Mueller,\nJeffrey Ladish, Joshua Landau, Kamal Ndousse, Kamile Lukosuite, Liane Lovitt,\nMichael S

We are using the **Recursive Character** Text Splitting technique here, which is the recommended go-to method in LangChain for general text.

Here is how it works:

* It tries to split the text using a strict hierarchy of separators, starting with double newlines (\n\n for paragraphs), then single newlines (\n), spaces ( ), and finally individual characters ("").

* The splitter attempts to keep related pieces of text (like entire paragraphs or sentences) together as much as possible before moving down to a smaller separator.

* It will only resort to breaking smaller units (like splitting a word by characters) if a single block is still larger than your chunk_size.

* The chunk_overlap acts as a safety net, carrying over a portion of text into the next chunk so you don't abruptly slice a sentence or concept in half!

Each chunk also has 'page_content' and 'metadata'. The chunk size is only applied to the page_content, the metadata is ignored during size calculation

### EMBEDDING and VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # since every record that we insert into the vector DB will have a unique ID and well be generating it using uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # going to load the all_MiniLM-L6-v2 model 

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}") # each chunks gets converted to a 384 dimensional vector by default
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: # takes a list of strings and returns a numpy array
        """
        Generate embeddings for a list of texts
    
        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initializr the embedding manager
embedding_manager= EmbeddingManager()
embedding_manager

"""
once we call `embedding_manager.generate_embeddings(chunks)`, all the document chunks are passed to `self.model.encode(texts)`. 
Here, **`all-MiniLM-L6-v2` is a pretrained SentenceTransformer model based on the SBERT (Sentence-BERT) approach**, using a lightweight MiniLM 
Transformer as its backbone; it tokenizes each chunk, processes the tokens through the Transformer to capture their contextual and semantic relationships, 
and then uses pooling to combine the token representations into **one fixed-size 384-dimensional vector per chunk**. Thus, if you have `N` chunks, you get 
an embedding matrix of shape **`(N, 384)`**.

"""

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


'\nonce we call `embedding_manager.generate_embeddings(chunks)`, all the document chunks are passed to `self.model.encode(texts)`. \nHere, **`all-MiniLM-L6-v2` is a pretrained SentenceTransformer model based on the SBERT (Sentence-BERT) approach**, using a lightweight MiniLM \nTransformer as its backbone; it tokenizes each chunk, processes the tokens through the Transformer to capture their contextual and semantic relationships, \nand then uses pooling to combine the token representations into **one fixed-size 384-dimensional vector per chunk**. Thus, if you have `N` chunks, you get \nan embedding matrix of shape **`(N, 384)`**.\n\n'

### VectorStore (VectorDB)

In [9]:
class VectorStore: 
    """Manages document embeddings in a ChromaDB vector store

    This class acts as a wrapper around chromaDB, its job is to: 
    Create/open a persistent vector database.
    Create/open a collection called pdf_documents.
    Store our chunks + embeddings + metadata.   
    
    """

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "data/vector_store" # actualy went in the agentic ai folder, instead of RAG lol :(
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
                                (store on the hard disk)
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )
            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents

        This is where our previously created chunks and their embeddings are prepared for storage.

        For every chunk, it creates:

        Unique ID
            +
        Chunk's text
            +
        Metadata
            +
        384-dimensional embedding
        """
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(
                embedding.tolist()
            )

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore= VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
### Extracts the actual text from every chunk and creates a list of strings.
texts=[doc.page_content for doc in chunks]

### Sends all those chunk texts to all-MiniLM-L6-v2 and generates one 384-dimensional embedding vector per chunk
embeddings=embedding_manager.generate_embeddings(texts)

### Stores each chunk + its embedding + metadata + unique ID in ChromaDB.
vectorstore.add_documents(chunks, embeddings)
'''
Overall: 
chunks
  ↓
extract text
  ↓
texts
  ↓
SentenceTransformer
  ↓
384-D vectors
  ↓
ChromaDB
  ↓
[chunk + vector + metadata]
'''

Generating embeddings for 1020 texts...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Generated embeddings with shape: (1020, 384)
Adding 1020 documents to vector store...
Successfully added 1020 documents to vector store
Total documents in collection: 1020


'\nOverall: \nchunks\n  ↓\nextract text\n  ↓\ntexts\n  ↓\nSentenceTransformer\n  ↓\n384-D vectors\n  ↓\nChromaDB\n  ↓\n[chunk + vector + metadata]\n'

### RAG PIPELINE (INJESTION + RETRIEVAL)
![Workflow](Images/rag_pipeline.png)
